In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: make output cleaner
pd.set_option('display.max_columns', None)


In [3]:
from google.colab import files
uploaded = files.upload()


Saving archive.zip to archive (1).zip


In [4]:
import zipfile

with zipfile.ZipFile("archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("extracted_data")


In [5]:
import os

os.listdir("extracted_data")


['Sample - Superstore.csv']

In [7]:
df = pd.read_csv(
    "extracted_data/Sample - Superstore.csv",
    encoding="latin1"
)

df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [8]:
df.shape
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [9]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df[['Order Date','Ship Date']].dtypes


,0
Order Date,datetime64[ns]
Ship Date,datetime64[ns]


In [11]:
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100
df[['Sales','Profit','Profit Margin %']].head()



,Sales,Profit,Profit Margin %
0,261.9600,41.9136,16.00
1,731.9400,219.5820,30.00
2,14.6200,6.8714,47.00
3,957.5775,-383.0310,-40.00
4,22.3680,2.5164,11.25


In [14]:
df['Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Processing Days'].describe()
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['YearMonth'] = df['Order Date'].dt.to_period('M')
monthly_sales = df.groupby('YearMonth')['Sales'].sum().reset_index()
monthly_sales.head()
customer_ltv = df.groupby('Customer Name')['Sales'].sum().reset_index()
customer_ltv = customer_ltv.sort_values(by='Sales', ascending=False)

customer_ltv.head(10)





,Customer Name,Sales
686,Sean Miller,25043.050
730,Tamara Chand,19052.218
622,Raymond Buch,15117.339
757,Tom Ashbrook,14595.620
6,Adrian Barton,14473.571
441,Ken Lonsdale,14175.229
671,Sanjit Chand,14142.334
334,Hunter Lopez,12873.298
672,Sanjit Engle,12209.438
156,Christopher Conant,12129.072


In [15]:
customer_profit = df.groupby('Customer Name').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Profit Margin %': 'mean'
}).reset_index()

customer_profit = customer_profit.sort_values(by='Sales', ascending=False)

customer_profit.head(10)


,Customer Name,Sales,Profit,Profit Margin %
686,Sean Miller,25043.050,-1980.7393,7.777778
730,Tamara Chand,19052.218,8981.3239,21.291667
622,Raymond Buch,15117.339,6976.0959,20.962963
757,Tom Ashbrook,14595.620,4703.7883,14.500000
6,Adrian Barton,14473.571,5444.8055,-8.312500
441,Ken Lonsdale,14175.229,806.8550,-3.256705
671,Sanjit Chand,14142.334,5757.4119,26.306818
334,Hunter Lopez,12873.298,5622.4292,31.409091
672,Sanjit Engle,12209.438,2650.6769,19.394737
156,Christopher Conant,12129.072,2177.0493,-1.075758


In [16]:
customer_profit.sort_values(by='Profit', ascending=False).head(10)


,Customer Name,Sales,Profit,Profit Margin %
730,Tamara Chand,19052.218,8981.3239,21.291667
622,Raymond Buch,15117.339,6976.0959,20.962963
671,Sanjit Chand,14142.334,5757.4119,26.306818
334,Hunter Lopez,12873.298,5622.4292,31.409091
6,Adrian Barton,14473.571,5444.8055,-8.312500
757,Tom Ashbrook,14595.620,4703.7883,14.500000
157,Christopher Martinez,8954.020,3899.8904,17.900000
431,Keith Dawkins,8181.256,3038.6254,24.519433
35,Andy Reiter,6608.448,2884.6208,38.444444
194,Daniel Raglin,8350.868,2869.0760,3.910256


In [17]:
customer_profit[customer_profit['Customer Name'] == 'Sean Miller']


,Customer Name,Sales,Profit,Profit Margin %
686,Sean Miller,25043.05,-1980.7393,7.777778


In [18]:
df[df['Customer Name'] == 'Sean Miller']['Discount'].mean()
df[df['Customer Name'] == 'Sean Miller'].groupby('Category').agg({
    'Sales':'sum',
    'Profit':'sum'
})
df[df['Customer Name'] == 'Sean Miller'].groupby('Sub-Category').agg({
    'Sales':'sum',
    'Profit':'sum'
}).sort_values(by='Profit')


,Sales,Profit
Sub-Category,,
Machines,23459.780,-1827.5044
Storage,663.072,-165.7680
Binders,99.588,-82.9900
Supplies,3.488,-0.6976
Art,8.016,1.0020
Accessories,21.728,3.8024
Fasteners,18.936,5.9175
Paper,88.872,30.5412
Furnishings,679.570,54.9576


In [19]:
df.groupby('Sub-Category').agg({
    'Sales':'sum',
    'Profit':'sum'
}).sort_values(by='Profit')


,Sales,Profit
Sub-Category,,
Tables,206965.5320,-17725.4811
Bookcases,114879.9963,-3472.5560
Supplies,46673.5380,-1189.0995
Fasteners,3024.2800,949.5182
Machines,189238.6310,3384.7569
Labels,12486.3120,5546.2540
Art,27118.7920,6527.7870
Envelopes,16476.4020,6964.1767
Furnishings,91705.1640,13059.1436


In [20]:
df.groupby('Sub-Category')['Discount'].mean().sort_values(ascending=False)
df[df['Customer Name']=='Sean Miller'].groupby('Sub-Category')['Discount'].mean()


,Discount
Sub-Category,
Accessories,0.200000
Art,0.200000
Binders,0.700000
Fasteners,0.200000
Furnishings,0.133333
Machines,0.500000
Paper,0.150000
Storage,0.200000
Supplies,0.200000


In [21]:
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100
df['Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Loss Flag'] = df['Profit'] < 0
df['High Discount Flag'] = df['Discount'] >= 0.4



In [22]:
region_summary = df.groupby('Region').agg({
    'Sales':'sum',
    'Profit':'sum'
}).reset_index()

category_summary = df.groupby('Sub-Category').agg({
    'Sales':'sum',
    'Profit':'sum'
}).reset_index()

monthly_summary = df.groupby('YearMonth').agg({
    'Sales':'sum',
    'Profit':'sum'
}).reset_index()


In [23]:
df.to_csv("Retail_Analytics_Final.csv", index=False)
